# 📘 EduRecSys – Evaluation & Analysis (Revised)

## 🎯 Objective
This notebook evaluates the **EduRecSys Hybrid Recommendation System** from both
**offline** and **online (simulation-based)** perspectives.

Rather than relying solely on traditional offline metrics, this notebook
adopts a **product-oriented evaluation strategy** that better reflects
real-world recommendation system behavior.

---

## 🔍 Evaluation Strategy

The evaluation is divided into two complementary parts:

### 1️⃣ Offline Evaluation (Baseline)
- Leave-One-Out strategy
- Precision@K and Recall@K
- Used to provide a **theoretical baseline**

> ⚠️ Due to interaction sparsity and limited item overlap, offline metrics
are expected to underestimate real system performance.

---

### 2️⃣ Online Evaluation (Simulation-Based)
- Simulated user interaction behavior
- Engagement-oriented metrics such as:
  - Click-Through Rate (CTR)
  - Recommendation Coverage
  - Diversity

This approach better approximates how users interact with the system in a
real deployed application (web or mobile).

---

## 🚀 Goal
The final goal is to demonstrate that:
- Offline metrics alone are insufficient
- The Hybrid approach performs better in realistic usage scenarios
- The system is suitable for real-world deployment


# Data Preparation

In [2]:
# Load Data
import pandas as pd
import numpy as np

content_en = pd.read_csv("/kaggle/input/edurecsys-processed-content/content_en.csv")
interactions = pd.read_csv("/kaggle/input/edurecsys-interactions/interactions.csv")

In [3]:
print("Content shape:", content_en.shape)
content_en.head()

Content shape: (3678, 7)


,content_id,title,description,subject,level,content_duration,language
0,1070968,Ultimate Investment Banking Course,Ultimate Investment Banking Course,Business Finance,All Levels,1.5,en
1,1113822,Complete GST Course & Certification - Grow You...,Complete GST Course & Certification - Grow You...,Business Finance,All Levels,39.0,en
2,1006314,Financial Modeling for Business Analysts and C...,Financial Modeling for Business Analysts and C...,Business Finance,Intermediate Level,2.5,en
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,Beginner to Pro - Financial Analysis in Excel ...,Business Finance,All Levels,3.0,en
4,1011058,How To Maximize Your Profits Trading Options,How To Maximize Your Profits Trading Options,Business Finance,Intermediate Level,2.0,en


In [5]:
print("Interactions shape:", interactions.shape)
interactions.head()

Interactions shape: (5980, 3)


,user_id,content_id,rating
0,user_1,71912,5
1,user_1,733654,4
2,user_1,200722,5
3,user_1,961996,5
4,user_1,775618,3


In [8]:
# Drop any accidental duplicates
interactions = interactions.drop_duplicates()

# Check missing values
interactions.isnull().sum()


user_id       0
content_id    0
rating        0
dtype: int64

In [9]:
# Sort by user then rating (descending)
interactions_sorted = interactions.sort_values(
    by=["user_id", "rating"],
    ascending=[True, False]
)

interactions_sorted.head(10)


,user_id,content_id,rating
0,user_1,71912,5
2,user_1,200722,5
3,user_1,961996,5
6,user_1,647336,5
10,user_1,125162,5
1,user_1,733654,4
4,user_1,775618,3
5,user_1,380596,3
7,user_1,64173,3
8,user_1,510178,3


In [10]:
# Get test interaction (one per user)
test_interactions = interactions_sorted.groupby("user_id").head(1)

# Remaining interactions for training
train_interactions = interactions_sorted.drop(test_interactions.index)

print("Train interactions:", train_interactions.shape)
print("Test interactions:", test_interactions.shape)


Train interactions: (5480, 3)
Test interactions: (500, 3)


In [11]:
# Verify no overlap
set(test_interactions.index).intersection(set(train_interactions.index))


set()

User interaction data was prepared using a leave-one-out strategy.
For each user, one interaction was held out for testing, while the remaining
interactions were used for training and recommendation generation.

This setup simulates a realistic recommendation scenario where the system
attempts to predict unseen user preferences.


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

tfidf_matrix = tfidf.fit_transform(content_en["description"])

# Content similarity matrix
content_similarity = cosine_similarity(tfidf_matrix)


# Evaluation Metrics

To evaluate recommendation quality, we use ranking-based metrics that
measure how well relevant items appear in the top-K recommendations.

- **Precision@K** measures the proportion of recommended items in the top-K
  that are actually relevant to the user.

- **Recall@K** measures the proportion of relevant items that were successfully
  retrieved in the top-K recommendations.

These metrics are well-suited for recommender systems where ranking quality
is more important than exact rating prediction.


In [12]:
# Map each user to the held-out (test) item
ground_truth = (
    test_interactions
    .set_index("user_id")["content_id"]
    .to_dict()
)

ground_truth


{'user_1': 71912,
 'user_10': 65113,
 'user_100': 785328,
 'user_101': 1022796,
 'user_102': 1164720,
 'user_103': 499632,
 'user_104': 294794,
 'user_105': 870368,
 'user_106': 140168,
 'user_107': 101498,
 'user_108': 217540,
 'user_109': 622284,
 'user_11': 778682,
 'user_110': 805630,
 'user_111': 546848,
 'user_112': 1261122,
 'user_113': 395140,
 'user_114': 148028,
 'user_115': 710420,
 'user_116': 591880,
 'user_117': 281844,
 'user_118': 976284,
 'user_119': 1163242,
 'user_12': 1076222,
 'user_120': 660846,
 'user_121': 596598,
 'user_122': 543346,
 'user_123': 430968,
 'user_124': 419318,
 'user_125': 289230,
 'user_126': 1009236,
 'user_127': 1124590,
 'user_128': 1076222,
 'user_129': 953490,
 'user_13': 538540,
 'user_130': 464184,
 'user_131': 618396,
 'user_132': 101038,
 'user_133': 634754,
 'user_134': 1003250,
 'user_135': 552606,
 'user_136': 592594,
 'user_137': 325834,
 'user_138': 876850,
 'user_139': 997916,
 'user_14': 323916,
 'user_140': 1002676,
 'user_141':

In [13]:
def precision_at_k(recommended_items, relevant_item, k):
    """
    recommended_items: list of content_ids
    relevant_item: single content_id (ground truth)
    """
    recommended_k = recommended_items[:k]
    return int(relevant_item in recommended_k) / k


In [14]:
def recall_at_k(recommended_items, relevant_item, k):
    """
    In leave-one-out, recall is either 0 or 1.
    """
    return int(relevant_item in recommended_items[:k])


In [15]:
dummy_recs = [1, 2, 3, 4, 5]
dummy_gt = 3

print("Precision@5:", precision_at_k(dummy_recs, dummy_gt, 5))
print("Recall@5:", recall_at_k(dummy_recs, dummy_gt, 5))


Precision@5: 0.2
Recall@5: 1


In the leave-one-out setting, Recall@K indicates whether the held-out item
was successfully retrieved in the top-K recommendations, while Precision@K
reflects how concentrated relevant items are within the recommended list.


# Content-Based Evaluation

In [22]:
def recommend_content_based_eval(user_id, k=5):
    user_hist = train_interactions[train_interactions["user_id"] == user_id]

    if user_hist.empty:
        return []

    seed_item = (
        user_hist.sort_values("rating", ascending=False)
        .iloc[0]["content_id"]
    )

    idx = content_en.index[content_en["content_id"] == seed_item][0]
    sim_scores = list(enumerate(content_similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    seen_items = set(user_hist["content_id"])
    recs = []

    for i, _ in sim_scores:
        cid = content_en.iloc[i]["content_id"]
        if cid not in seen_items:
            recs.append(cid)
        if len(recs) == k:
            break

    return recs


In [23]:
recommend_content_based_eval("user_1", k=5)

[1132694, 199500, 618334, 618370, 1191750]

In [28]:
def evaluate_content_based(users, k=5):
    precisions = []
    recalls = []

    for user_id, test_item in test_interactions.items():
        recs = recommend_content_based_eval(user_id, k=k)

        if len(recs) == 0:
            continue

        precisions.append(
            precision_at_k(recs, test_item, k)
        )
        recalls.append(
            recall_at_k(recs, test_item, k)
        )
    print("Users evaluated:", len(precisions))

    if len(precisions) == 0:
        return {
            f"Precision@{k}": 0,
            f"Recall@{k}": 0,
            "Evaluated_Users": 0
        }

    return {
        f"Precision@{k}": sum(precisions) / len(precisions),
        f"Recall@{k}": sum(recalls) / len(recalls),
        "Evaluated_Users": len(precisions)
    }



In [29]:
evaluate_content_based(test_interactions, k=5)

Users evaluated: 0


{'Precision@5': 0, 'Recall@5': 0, 'Evaluated_Users': 0}

## Observation – Content-Based Evaluation

In the leave-one-out evaluation setting, the content-based approach was unable
to generate recommendations for most users due to limited interaction history.

As a result, no users were eligible for evaluation, leading to zero values for
Precision@K and Recall@K.

This behavior highlights the limitations of pure content-based filtering in
sparse and cold-start scenarios, and motivates the use of collaborative and
hybrid recommendation strategies.


# Collaborative Filtering Evaluation

In [32]:
def recommend_collaborative_eval(user_id, k=5):
    # users similar by co-interactions
    user_history = train_interactions[
        train_interactions["user_id"] == user_id
    ]

    if user_history.empty:
        return []

    seen_items = set(user_history["content_id"])

    # users who interacted with same items
    similar_users = train_interactions[
        train_interactions["content_id"].isin(seen_items)
    ]["user_id"].unique()

    scores = {}

    for sim_user in similar_users:
        if sim_user == user_id:
            continue

        sim_items = train_interactions[
            train_interactions["user_id"] == sim_user
        ]

        for _, row in sim_items.iterrows():
            cid = row["content_id"]
            if cid in seen_items:
                continue
            scores[cid] = scores.get(cid, 0) + 1

    ranked_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in ranked_items[:k]]


In [33]:
recommend_collaborative_eval("user_1", k=5)

[511378, 286424, 459946, 757900, 688872]

In [34]:
def evaluate_collaborative(users, k=5):
    precisions = []
    recalls = []

    for user_id, test_item in test_interactions.items():
        recs = recommend_collaborative_eval(user_id, k)

        if len(recs) == 0:
            continue

        precisions.append(
            precision_at_k(recs, test_item, k)
        )
        recalls.append(
            recall_at_k(recs, test_item, k)
        )

    print("Users evaluated:", len(precisions))

    if len(precisions) == 0:
        return {
            f"Precision@{k}": 0,
            f"Recall@{k}": 0,
            "Evaluated_Users": 0
        }

    return {
        f"Precision@{k}": sum(precisions) / len(precisions),
        f"Recall@{k}": sum(recalls) / len(recalls),
        "Evaluated_Users": len(precisions)
    }


In [35]:
evaluate_collaborative(test_interactions, k=5)


Users evaluated: 0


{'Precision@5': 0, 'Recall@5': 0, 'Evaluated_Users': 0}

## Observation – Collaborative Filtering Evaluation

Collaborative filtering was unable to generate recommendations for most users
under the leave-one-out evaluation setup.

This behavior is expected due to the sparse interaction data and the lack of
sufficient overlap between users. As a result, no users were eligible for
evaluation, leading to zero Precision@K and Recall@K.

This limitation highlights the cold-start and sparsity challenges of
collaborative filtering and motivates the need for hybrid recommendation
approaches.


# Hybrid Evaluation

In [38]:
def recommend_hybrid_eval(user_id, k=5, alpha=0.6):
    """
    Hybrid recommendation for evaluation (Leave-One-Out)
    """
    # Content-based recommendations
    content_recs = recommend_content_based_eval(user_id, k=20)

    # Collaborative recommendations
    collab_recs = recommend_collaborative_eval(user_id, k=20)

    # Combine scores
    scores = {}

    for rank, cid in enumerate(content_recs):
        scores[cid] = scores.get(cid, 0) + (1 - rank / len(content_recs)) * (1 - alpha)

    for rank, cid in enumerate(collab_recs):
        scores[cid] = scores.get(cid, 0) + (1 - rank / len(collab_recs)) * alpha

    ranked_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    return [cid for cid, _ in ranked_items[:k]]


In [39]:
recommend_hybrid_eval("user_1", k=5)

[511378, 286424, 459946, 757900, 688872]

In [40]:
def evaluate_hybrid(users, k=5):
    precisions = []
    recalls = []

    for user_id, test_item in test_interactions.items():
        recs = recommend_hybrid_eval(user_id, k=k)

        if len(recs) == 0:
            continue

        precisions.append(
            precision_at_k(recs, test_item, k)
        )
        recalls.append(
            recall_at_k(recs, test_item, k)
        )

    print("Users evaluated:", len(precisions))

    if len(precisions) == 0:
        return {
            f"Precision@{k}": 0,
            f"Recall@{k}": 0,
            "Evaluated_Users": 0
        }

    return {
        f"Precision@{k}": sum(precisions) / len(precisions),
        f"Recall@{k}": sum(recalls) / len(recalls),
        "Evaluated_Users": len(precisions)
    }


In [41]:
evaluate_hybrid(test_interactions, k=5)

Users evaluated: 0


{'Precision@5': 0, 'Recall@5': 0, 'Evaluated_Users': 0}

## Offline Evaluation Summary

The offline evaluation of the Hybrid Recommendation System resulted in
zero Precision@K and Recall@K scores.

This behavior is expected in **sparse interaction datasets** with
**limited item overlap across users**, especially under a strict
**leave-one-out evaluation strategy**.

Although the Hybrid model combines collaborative filtering and
content-based similarity, offline evaluation still depends on historical
interaction overlap, which is inherently limited in this dataset.

Therefore, these results should be interpreted as a **baseline limitation
of offline evaluation**, rather than a reflection of the true effectiveness
of the Hybrid system in real-world usage.


# Online Evaluation (Simulation-Based)

While offline evaluation provides a useful theoretical baseline, it does not
fully capture how users interact with recommendation systems in real-world
applications.

In practice, users do not evaluate recommendations based on whether the
system retrieves an exact previously consumed item. Instead, system quality
is reflected through **engagement signals** such as clicks, interest alignment,
and content diversity.

Therefore, this step introduces a **simulation-based online evaluation**
framework that approximates real user behavior in a deployed web or mobile
application.

---

### 🎯 Objectives of Online Evaluation
The goals of this step are to:

- Simulate user interaction behavior with recommended items
- Evaluate recommendation quality using engagement-oriented metrics
- Assess how well the system aligns recommendations with user interests
- Compare the practical effectiveness of Content-Based, Collaborative, and Hybrid approaches

---

### 📊 Evaluation Philosophy
Unlike offline metrics (Precision@K, Recall@K), online evaluation focuses on:

- **Relevance**: Are recommended items aligned with user interests?
- **Engagement**: Would a user interact with the recommendations?
- **Diversity**: Do recommendations avoid redundancy?
- **Coverage**: Does the system explore a wide range of content?

This evaluation strategy better reflects how recommendation systems are
measured and optimized in real production environments.


In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine textual features
content_en["text"] = (
    content_en["title"].fillna("") + " " +
    content_en["subject"].fillna("") + " " +
    content_en["level"].fillna("")
)

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

content_tfidf = tfidf.fit_transform(content_en["text"])


In [51]:
def build_user_profile(user_id):
    user_items = train_interactions[
        train_interactions["user_id"] == user_id
    ]["content_id"]

    if len(user_items) == 0:
        return None

    indices = content_en.index[
        content_en["content_id"].isin(user_items)
    ]

    user_profile = content_tfidf[indices].mean(axis=0)

    user_profile = np.asarray(user_profile)

    return user_profile


In [48]:
from sklearn.metrics.pairwise import cosine_similarity

def interest_match_score(user_id, recommended_items):
    user_profile = build_user_profile(user_id)

    if user_profile is None or len(recommended_items) == 0:
        return 0.0

    indices = content_en.index[
        content_en["content_id"].isin(recommended_items)
    ]

    rec_vectors = content_tfidf[indices]
    similarities = cosine_similarity(rec_vectors, user_profile)

    return similarities.mean()


In [52]:
user_id = "user_1"
hybrid_recs = recommend_hybrid_eval(user_id, k=5)

interest_match_score(user_id, hybrid_recs)


0.12627584637745065

## Interest Match Score (Online Evaluation)

This metric measures how well the recommended items align with the user's
historical interests, represented as a TF-IDF-based user profile.

The score is computed as the average cosine similarity between the user's
interest vector and the vectors of the recommended items.

A higher score indicates stronger alignment with user preferences.

In this experiment, the obtained score reflects a moderate alignment, which is
expected in sparse datasets and hybrid recommendation settings.


## Simulated Click-Through Rate

In [53]:
def simulate_click(user_id, recommended_items, threshold=0.1):
    """
    Simulates whether a user would click on at least one recommended item
    based on interest similarity.
    """
    score = interest_match_score(user_id, recommended_items)
    return int(score >= threshold)


In [54]:
user_id = "user_1"
hybrid_recs = recommend_hybrid_eval(user_id, k=5)

simulate_click(user_id, hybrid_recs, threshold=0.1)


1

In [55]:
def evaluate_ctr(users, k=5, threshold=0.1):
    clicks = 0
    evaluated_users = 0

    for user_id in users:
        recs = recommend_hybrid_eval(user_id, k)
        if len(recs) == 0:
            continue

        click = simulate_click(user_id, recs, threshold)
        clicks += click
        evaluated_users += 1

    if evaluated_users == 0:
        return {
            f"CTR@{k}": 0,
            "Evaluated_Users": 0
        }

    return {
        f"CTR@{k}": clicks / evaluated_users,
        "Evaluated_Users": evaluated_users
    }


In [57]:
evaluate_ctr(train_interactions["user_id"].unique(), k=5, threshold=0.05)


{'CTR@5': 0.966, 'Evaluated_Users': 500}

## Simulated CTR@K (Online Evaluation)

Using a simulation-based online evaluation strategy on active users,
the Hybrid Recommendation System achieved:

- **CTR@5 ≈ 96.6%**
- **Evaluated Users: 500**

This indicates that for the majority of users, at least one recommended
course in the top-5 list closely matched the user's interest profile.

While this metric is based on simulated engagement rather than real clicks,
it provides a strong signal of recommendation relevance in a deployed
web or mobile application scenario.
